# Coffee RAG · v6 evaluation on Kaggle

Chạy trọn bộ thí nghiệm v6 (4 cấu hình × N run + soft recall) bằng Anthropic API, GPU Kaggle cho embedding.

**Trước khi Run:**
1. Settings (bên phải) → **Accelerator: GPU T4 x2** (hoặc P100), **Internet: On**, **Persistence: Files only** (tuỳ chọn).
2. **Add Input** → Datasets → chọn dataset bạn đã upload (tạo bằng `python -m evaluation.kaggle.pack_data`). Tên mount lấy theo panel Input bên phải; ô cấu hình tự dò nếu khai sai.
3. **Add-ons → Secrets**: thêm `ANTHROPIC_API_KEY`. Tuỳ chọn: `HF_TOKEN` (tắt warning Hugging Face), `GITHUB_TOKEN` nếu repo private.
4. Để **nối tiếp** phiên trước: Add Input → Your Work → chọn output của notebook lần trước (thư mục `results_v6`). Script tự copy và bỏ qua case đã chấm.
5. Chạy bằng **Save Version → Save & Run All (Commit)** để phiên chạy nền tối đa 12 giờ, không phụ thuộc trình duyệt.


In [ ]:
# ── Cấu hình ───────────────────────────────────────────────────
REPO_URL   = "https://github.com/hairbui76/coffee-rag.git"
BRANCH     = "main"
DATA_INPUT = "/kaggle/input/datasets/hairbui76/coffee-rag-upload"  # xem bằng: !find /kaggle/input -name ragas_eval_dataset_v6.json (tự dò nếu sai)

ONLY    = "all"      # all | bge | minilm   (tách phiên khi cần dưới 12 giờ)
RUNS    = "1 2 3"    # số lần lặp mỗi cấu hình, ví dụ "1 2 3 4 5"
WORKERS = "8"        # Build tier (5M token/phút) chịu được ~32 request đồng thời; giảm còn 4 nếu thấy nhiều 429

PIPELINE_MODEL = "claude-haiku-5-5"    # entity extraction + sinh câu trả lời
JUDGE_MODEL    = "claude-sonnet-5-5"   # judge RAGAS (mạnh hơn generator)
SMOKE_CASES    = 3                      # chạy thử trước khi chạy cả bộ; 0 để bỏ qua


In [ ]:
# ── Clone repo + cài thư viện ───────────────────────────────────
import os, subprocess, sys, shutil, glob
from pathlib import Path

WORK = Path("/kaggle/working"); REPO = WORK / "coffee-rag"
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try: return secrets.get_secret(name)
    except Exception: return ""

clone_url = REPO_URL
tok = secret("GITHUB_TOKEN")
if tok and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://github.com/", f"https://{tok}@github.com/")
if not REPO.exists():
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")   # dữ liệu lấy từ Dataset, không kéo LFS
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, str(REPO)], check=True, env=env)
os.chdir(REPO)
print("repo:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Chỉ cài những gì cần, không đụng torch có sẵn của Kaggle (đã có CUDA).
# langchain-community<0.4: bản 0.4 bỏ ChatVertexAI mà ragas 0.4.3 còn import.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "ragas>=0.4.3,<0.5", "langchain-community<0.4", "faiss-cpu", "rank-bm25", "langchain-text-splitters", "openai", "anthropic",
    "python-dotenv", "sentence-transformers", "pyarrow", "scipy", "tabulate", "matplotlib", "seaborn", "pydantic>=2"],
    check=True)
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
# ── Nối dữ liệu từ Dataset, khôi phục kết quả phiên trước ─────────
data_in = Path(DATA_INPUT)
if not (data_in / "ragas_eval_dataset_v6.json").exists():
    # Tên mount phụ thuộc tên Dataset bạn đặt; tự dò theo file đặc trưng.
    found = glob.glob("/kaggle/input/**/ragas_eval_dataset_v6.json", recursive=True)
    if not found:
        print("Cây thư mục /kaggle/input:"); subprocess.run(["find", "/kaggle/input", "-maxdepth", "4"], check=False)
    assert found, "Không thấy ragas_eval_dataset_v6.json dưới /kaggle/input. Add Input → Dataset đã upload bằng pack_data.py, hoặc restart session sau khi Add."
    data_in = Path(found[0]).parent
    print(f"DATA_INPUT không tồn tại, dùng {data_in}")
assert (data_in / "data" / "embeddings" / "beans.index").exists(), f"Dataset ở {data_in} thiếu data/embeddings/beans.index"

for rel in ["data/processed", "data/embeddings"]:
    dst = REPO / rel
    if dst.is_symlink() or dst.exists():
        shutil.rmtree(dst) if dst.is_dir() and not dst.is_symlink() else dst.unlink()
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.symlink_to(data_in / rel)
shutil.copy2(data_in / "ragas_eval_dataset_v6.json", REPO / "ragas_eval_dataset_v6.json")

RESULTS = REPO / "evaluation" / "results" / "v6"; RESULTS.mkdir(parents=True, exist_ok=True)
restored = 0
for prev in glob.glob("/kaggle/input/**/results_v6", recursive=True):   # output của notebook lần trước
    for f in Path(prev).glob("*"):
        if f.is_file():
            shutil.copy2(f, RESULTS / f.name); restored += 1
    mini = Path(prev).parent / "embeddings_minilm"
    if mini.exists() and not (REPO / "data" / "embeddings_minilm").exists():
        shutil.copytree(mini, REPO / "data" / "embeddings_minilm")
        print("restored MiniLM index from", mini)
print(f"restored {restored} result files from previous sessions")
print("v6 cases:", len(__import__('json').load(open(REPO / 'ragas_eval_dataset_v6.json', encoding='utf-8'))))


In [ ]:
# ── .env từ Kaggle Secrets ──────────────────────────────────────
key = secret("ANTHROPIC_API_KEY")
assert key, "Thêm ANTHROPIC_API_KEY trong Add-ons → Secrets rồi chạy lại."
(REPO / ".env").write_text(f"""LLM_PROVIDER=anthropic
ANTHROPIC_API_KEY={key}
ANTHROPIC_MODEL={PIPELINE_MODEL}
EMBEDDING_MODEL=BAAI/bge-m3
TOP_K_BEANS=5
TOP_K_NEWS=5
RAGAS_JUDGE_PROVIDER=anthropic
RAGAS_EVALUATOR_MODEL={JUDGE_MODEL}
RAGAS_AR_EMBEDDING_MODEL=BAAI/bge-m3
RAGAS_DO_NOT_TRACK=true
DEBUG=false
""")
os.environ.update({"RAGAS_DO_NOT_TRACK": "true", "TOKENIZERS_PARALLELISM": "false", "HF_HUB_DISABLE_PROGRESS_BARS": "1"})
hf = secret("HF_TOKEN")          # tuỳ chọn: tắt warning "unauthenticated requests to the HF Hub", tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
    with open(REPO / ".env", "a") as f:
        f.write("HF_TOKEN=" + hf + chr(10))
print(".env written (keys hidden)", "| HF_TOKEN:", "set" if hf else "not set")


In [ ]:
# ── Chạy thử vài case để bắt lỗi trước khi chạy cả bộ ───────────
def stream(cmd, env_extra=None):
    env = dict(os.environ, **(env_extra or {}))
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"exit {p.returncode}: {' '.join(map(str, cmd))}")

if SMOKE_CASES:
    stream([sys.executable, "-m", "evaluation.ragas_eval", "--dataset", "ragas_eval_dataset_v6.json",
            "--mode", "full", "--limit", str(SMOKE_CASES), "--out", str(RESULTS / "smoke.csv"),
            "--run-id", "smoke", "--seed", "0", "--judge-temperature", "0", "--workers", "2", "-v"],
           {"EMBEDDING_MODEL": "BAAI/bge-m3", "RAGAS_AR_EMBEDDING_MODEL": "BAAI/bge-m3"})
    import pandas as pd
    df = pd.read_csv(RESULTS / "smoke.csv")
    print(df[[c for c in ["id","context_precision","context_recall","faithfulness","answer_relevancy","recall_at_k","elapsed_s","error"] if c in df]].to_string())
    assert df["error"].fillna("").eq("").all(), "Có lỗi trong smoke test, xem cột error ở trên"


In [ ]:
# ── Chạy cả bộ (resumable). Có thể mất nhiều giờ. ───────────────
import time; t0 = time.time()
stream(["bash", "evaluation/run_v6.sh"], {"ONLY": ONLY, "RUNS": RUNS, "WORKERS": WORKERS, "OUT": "evaluation/results/v6"})
print(f"\nDone in {(time.time()-t0)/3600:.1f} h")


In [ ]:
# ── Đóng gói kết quả vào /kaggle/working để Save Version giữ lại ──
out_dir = WORK / "results_v6"
if out_dir.exists(): shutil.rmtree(out_dir)
shutil.copytree(RESULTS, out_dir)
sample = REPO / "evaluation" / "annotation" / "annotation_sample.json"
if sample.exists(): shutil.copy2(sample, out_dir / "annotation_sample.json")
mini = REPO / "data" / "embeddings_minilm"
if mini.exists() and not mini.is_symlink():
    if (WORK / "embeddings_minilm").exists(): shutil.rmtree(WORK / "embeddings_minilm")
    shutil.copytree(mini, WORK / "embeddings_minilm")
shutil.make_archive(str(WORK / "coffee_rag_results_v6"), "zip", root_dir=out_dir)
# Không giữ repo (có .env chứa key) trong output
shutil.rmtree(REPO, ignore_errors=True)
print("Output files:")
for f in sorted(WORK.rglob("*")):
    if f.is_file() and f.suffix in (".csv", ".zip", ".md", ".json", ".tex"):
        print(" ", f.relative_to(WORK), f"{f.stat().st_size/1e6:.1f} MB")
